# Black-Scholes Options Pricing
Pricing European call and put options and backing out implied volatility from market prices.

In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq

In [2]:
def black_scholes_call(S, K, r, sigma, T):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    call = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    return call

def black_scholes_put(S, K, r, sigma, T):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    put = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    return put

## Implied Volatility — Brent's Method
Volatility is the only input to Black-Scholes not directly observable from the market.
We back it out by finding the sigma that makes BS price = market price.

Used Brent's method instead of Newton-Raphson because when options are deep ITM/OTM, Vega approaches zero — Newton-Raphson divides by Vega and becomes unstable. Brent's method is derivative-free so it handles this cleanly.

In [3]:
def implied_volatility(C_market, S, K, r, T, option_type='call'):
    pricing_fn = black_scholes_call if option_type == 'call' else black_scholes_put

    def objective(sigma):
        return pricing_fn(S, K, r, sigma, T) - C_market

    try:
        iv = brentq(objective, 1e-6, 5.0)
        return iv
    except ValueError:
        return None

In [4]:
# NSE example — ICICI Bank
S = 986.70
K = 990
r = 0.07
T = 0.0381
C_market = 14.50

iv = implied_volatility(C_market, S, K, r, T)
print(f"Implied Volatility : {iv:.1%}")
print(f"BS Price check     : {black_scholes_call(S, K, r, iv, T):.2f}")
print(f"Market Price       : {C_market}")

Implied Volatility : 19.3%
BS Price check     : 14.50
Market Price       : 14.5
